In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('england_master.csv')
df['date'] = pd.PeriodIndex(df['Unnamed: 0'], freq='Q').to_timestamp()
df.set_index('date', inplace=True)

df_clean = df[['starts', 'hprice', 'cc', 'rate', 'vol']].dropna()


In [ ]:
df_clean['log_starts'] = np.log(df_clean['starts'])
df_clean['log_hprice'] = np.log(df_clean['hprice'])
df_clean['log_cc'] = np.log(df_clean['cc'])

In [ ]:
#Profit margin
df_clean['margin'] = df_clean['log_hprice'] - df_clean['log_cc']


y = df_clean['log_starts']
X = df_clean[['margin', 'rate']]
X = sm.add_constant(X)


In [ ]:
#VIF
vif_table = pd.DataFrame()
vif_table["Variable"] = X.columns
vif_table["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print("VIF Results")
print(vif_table)

In [ ]:
#variables for RF
df_clean['starts_lag1'] = df_clean['log_starts'].shift(1)
df_clean['starts_lag4'] = df_clean['log_starts'].shift(4)
df_clean['vol_lag1'] = df_clean['vol'].shift(1)
df_clean['rate_lag1'] = df_clean['rate'].shift(1)
df_clean['time_trend'] = np.arange(len(df_clean))

df_gtvp = df_clean.dropna()
y_gtvp = df_gtvp['log_starts']
X_gtvp = sm.add_constant(df_gtvp[['margin', 'rate']])
S = df_gtvp[['starts_lag1', 'starts_lag4', 'vol_lag1', 'rate_lag1', 'time_trend']]

#GTVP
rf = RandomForestRegressor(n_estimators=500, min_samples_leaf=15, random_state=42)
rf.fit(S, y_gtvp)
leaf_assignments = rf.apply(S)

valid_idx = df_gtvp.index
gtvps = pd.DataFrame(index=valid_idx, columns=X_gtvp.columns, dtype=float)

#WLS

for t_idx in range(len(valid_idx)):
    current_leaves = leaf_assignments[t_idx, :]
    weights = np.sum(leaf_assignments == current_leaves, axis=1)
    weights = weights / weights.sum()

    wls_model = sm.WLS(y_gtvp, X_gtvp, weights=weights).fit()
    gtvps.iloc[t_idx] = wls_model.params

In [ ]:
#Plot
plt.figure(figsize=(12, 6))
plt.plot(gtvps.index, gtovp_margin := gtvps['margin'], color='darkgreen', linewidth=2.5)
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.title('GTVP: Time-Varying Elasticity of Profit Margin ', fontsize=14, fontweight='bold')
plt.ylabel('Elasticity Coefficient', fontsize=12)
plt.xlabel('Date', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plots/gtvp_margin.png')
plt.show()

print(f"Mean Margin Elasticity: {gtvps['margin'].mean():.4f}")
print(f"Mean Rate Elasticity: {gtvps['rate'].mean():.4f}")